[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/02-data-science/01_numpy-basics.ipynb)

*Python for Data Science · Notebook 1 of 8*

# NumPy Basics

Arrays: creating them, their shape and dtype, indexing, slicing, reshaping and fancy indexing.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** NumPy is a spreadsheet grid for Python: every cell has the same type, so the computer can race through it.

**What interviewers really test here:** Shape, dtype, views vs copies, and why arrays beat lists for numeric work.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why NumPy? *(Basics)*
- [ ] 2. Creating arrays *(Basics)*
- [ ] 3. How arrays are stored, and why dtype matters *(Basics)*
- [ ] 4. Shape, dimensions and dtype *(Core)*
- [ ] 5. Indexing and slicing *(Core)*
- [ ] 6. Reading 2-D slices, worked examples *(Core)*
- [ ] 7. Views and copies: a bug worth knowing *(Core)*
- [ ] 8. Boolean masks and fancy indexing *(Advanced)*
- [ ] 9. Reshaping and combining *(Advanced)*
- [ ] 10. Masks, where and -1, step by step *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### What NumPy is

**NumPy** (Numerical Python) is the fundamental library for numerical computing in Python. Its core object is the **ndarray** — an N-dimensional, fixed-size, **homogeneous** array (all elements have the same data type) stored in one contiguous block of memory.



Operations on ndarrays are implemented in C and applied to whole arrays at once (**vectorisation**), which avoids the overhead of Python loops. NumPy is the base of the scientific Python stack: pandas, SciPy, Matplotlib, scikit-learn, TensorFlow and PyTorch all use or mirror its arrays.

### ndarray vs Python list

|   | Python list | NumPy ndarray |
| --- | --- | --- |
| Element types | any mix | one dtype |
| Memory | array of pointers to separate objects | one contiguous buffer of raw values |
| Size | grows with append | fixed (creating a bigger one copies) |
| Arithmetic | `+` concatenates, `*` repeats | element-wise maths |
| Speed for maths | slow (interpreted loop) | fast (compiled, SIMD) |
| Multi-dimensional | lists of lists (ragged allowed) | true N-D with shape |

### Array attributes

| Attribute | Meaning | For `np.zeros((2, 3))` |
| --- | --- | --- |
| `ndim` | number of axes | 2 |
| `shape` | length of each axis | (2, 3) |
| `size` | total elements | 6 |
| `dtype` | element type | float64 |
| `itemsize` | bytes per element | 8 |
| `nbytes` | total bytes | 48 |
| `T` | transpose | shape (3, 2) |



**Axes**: axis 0 runs down the rows, axis 1 runs across the columns. Many functions take `axis=` to choose which direction to work along.

### Data types (dtype)

| Kind | dtypes |
| --- | --- |
| Integers | `int8, int16, int32, int64`, unsigned `uint8 …` |
| Floats | `float16, float32, float64` (default) |
| Complex | `complex64, complex128` |
| Boolean | `bool` |
| Text | `<U10` (Unicode, fixed max length) |
| Dates | `datetime64`, `timedelta64` |
| Anything | `object` (Python objects — loses the speed benefit) |



**Upcasting**: when types mix, NumPy picks a type that holds all values: int + float → float. **Overflow**: fixed-size ints wrap around silently (an `int8` 127 + 1 becomes −128). Missing values are usually `np.nan`, which is a float.

### Indexing rules

| Kind | Syntax | Returns |
| --- | --- | --- |
| Basic index | `a[i]`, `m[i, j]` | a scalar or a lower-dimensional view |
| Slice | `a[start:stop:step]`, `m[:, 1:3]` | **view** (shares memory) |
| Boolean mask | `a[a > 0]` | **copy** (always 1-D for a full-shape mask) |
| Fancy (integer array) | `a[[0, 2, 4]]` | **copy** |
| Ellipsis / new axis | `a[..., 0]`, `a[:, np.newaxis]` | view |



Element-wise logical operators for masks: `&` (and), `|` (or), `~` (not), with each comparison in parentheses because `&` binds tighter than `>`.

### Views vs copies

Slicing, reshaping (when possible), `ravel` and `.T` return **views**: new array objects that look at the same memory. Writing to a view changes the original. This makes slicing O(1) and memory-cheap.



Use `.copy()` for an independent array. Check with `np.shares_memory(a, b)` or `b.base is a`.

---
# Part 2: Lessons, step by step

## Lesson 1: Why NumPy?

*Level: Basics*

Python lists can hold anything, but that flexibility makes maths on them slow: every item is a separate Python object and a loop handles one at a time. **NumPy** stores numbers in one compact block of memory, all of the same type, and runs operations on the whole block in fast compiled code.



```python
import numpy as np          # the standard way to import it
prices = np.array([10, 20, 30])
prices * 1.18               # array([11.8, 23.6, 35.4]) — no loop
```



That is called **vectorisation**: you write the operation once and it applies to every element. It is typically 10–100× faster than a Python loop and much shorter to read. Pandas, Matplotlib, Seaborn and scikit-learn are all built on NumPy arrays.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def why_numpy():
    prices = [10, 20, 30]
    with_tax = []
    for p in prices:
        with_tax.append(p * 1.18)
    arr = np.array(prices)
    arr_with_tax = arr * 1.18
    print(with_tax)
    print(arr_with_tax)


why_numpy()

[11.799999999999999, 23.599999999999998, 35.4]
[11.8 23.6 35.4]


**✏️ Your turn:** you now have what you need for [Q1](#q1). Try one now, or carry on and do them in Part 3.

## Lesson 2: Creating arrays

*Level: Basics*

| Code | Gives |
| --- | --- |
| `np.array([1, 2, 3])` | from a list |
| `np.array([[1, 2], [3, 4]])` | 2-D array from a nested list |
| `np.zeros(5)`, `np.ones((2, 3))` | filled with 0 / 1 (floats by default) |
| `np.full((2, 2), 7)` | filled with a value |
| `np.arange(0, 10, 2)` | like range: 0 2 4 6 8 |
| `np.linspace(0, 1, 5)` | 5 evenly spaced numbers, both ends included |
| `np.eye(3)` | identity matrix |
| `np.random.default_rng(0).integers(1, 7, 10)` | 10 random dice rolls (seeded, so repeatable) |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def creating_demo():
    a = np.array([1, 2, 3])
    zeros = np.zeros(4)
    grid = np.ones((2, 3), dtype=int)
    evens = np.arange(0, 10, 2)
    steps = np.linspace(0, 1, 5)
    rolls = np.random.default_rng(0).integers(1, 7, 6)
    print(a, zeros, evens, steps, rolls, sep="\n")


creating_demo()

[1 2 3]
[0. 0. 0. 0.]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[6 4 4 2 2 1]


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 3: How arrays are stored, and why dtype matters

*Level: Basics*

A Python list holds pointers to separate objects scattered around memory. A NumPy array holds raw numbers side by side, **all of the same type**. That's why it's so fast and compact.



| dtype | Bytes each | Range or use | 1,000 values |
| --- | --- | --- | --- |
| int8 | 1 | −128 to 127 | 1,000 bytes |
| int64 | 8 | very large whole numbers | 8,000 bytes |
| float64 | 8 | decimals (the default) | 8,000 bytes |



#### NumPy picks one type for everything



- `np.array([1, 2.5])` becomes all floats: [1.0, 2.5].
- `np.array([1, 2.5, "3"])` becomes all **text**. Maths now fails, a classic surprise when loading messy data.



#### Small types can overflow silently



`np.array([127], dtype=np.int8) + 1` gives **−128**, with no error. The value wraps around the end of the range. Use a smaller type only when you know the values fit.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def dtype_demo():
    mixed_numbers = np.array([1, 2.5])
    mixed_text = np.array([1, 2.5, "3"])
    small = np.zeros(1000, dtype=np.int8).nbytes
    large = np.zeros(1000, dtype=np.int64).nbytes
    wrapped = np.array([127], dtype=np.int8) + 1
    converted = np.array(["1", "2"]).astype(int)


dtype_demo()

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 4: Shape, dimensions and dtype

*Level: Core*

Every array has:




- `ndim` — number of dimensions (axes): 1 for a row of numbers, 2 for a table.
- `shape` — length along each axis: `(3,)` for 3 numbers, `(2, 3)` for 2 rows × 3 columns.
- `size` — total number of elements.
- `dtype` — the one type every element shares: `int64`, `float64`, `bool`, `<U5` (text)…



Mixing types upgrades everything to a common type: `np.array([1, 2.5])` is all floats. Convert with `arr.astype(int)`. In the animations, each array's dtype and shape appear in its label.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def shape_demo():
    table = np.array([[1, 2, 3], [4, 5, 6]])
    rows, cols = table.shape
    mixed = np.array([1, 2.5, 3])
    as_int = mixed.astype(int)
    print(table.ndim, table.shape, table.size, table.dtype)
    print(mixed.dtype, as_int, as_int.dtype)


shape_demo()

2 (2, 3) 6 int64
float64 [1 2 3] int64


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 5: Indexing and slicing

*Level: Core*

1-D arrays index like lists. 2-D arrays take **row, column** inside one pair of brackets:



```python
m = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
m[1, 2]        # 6        row 1, column 2
m[0]           # [1 2 3]  the whole first row
m[:, 0]        # [1 4 7]  every row, column 0
m[:2, 1:]      # [[2 3] [5 6]]  a block
m[::-1]        # rows in reverse order
```



**⚠️ Watch out:** A slice is a **view**: it shares memory with the original, so changing the slice changes the original. Use `m[:2].copy()` when you need an independent array.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def slicing_demo():
    m = np.arange(1, 10).reshape(3, 3)
    cell = m[1, 2]
    first_col = m[:, 0]
    block = m[:2, 1:]
    view = m[0]
    view[0] = 100
    print(cell, first_col, block, m, sep="\n")


slicing_demo()

6
[100   4   7]
[[2 3]
 [5 6]]
[[100   2   3]
 [  4   5   6]
 [  7   8   9]]


**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q6](#q6), [Q9](#q9), [Q7](#q7), [Q8](#q8), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 6: Reading 2-D slices, worked examples

*Level: Core*

Take `m = np.arange(12).reshape(3, 4)`:



```python
[[ 0,  1,  2,  3],
 [ 4,  5,  6,  7],
 [ 8,  9, 10, 11]]
```



Read `m[rows, columns]`, with a comma between the two.



| Code | Means | Result | Shape |
| --- | --- | --- | --- |
| `m[1, 2]` | row 1, column 2 | 6 | single value |
| `m[0]` | row 0 | [0, 1, 2, 3] | (4,) |
| `m[:, 1]` | column 1 | [1, 5, 9] | (3,) |
| `m[1:, 2:]` | rows from 1, columns from 2 | [[6, 7], [10, 11]] | (2, 2) |
| `m[::2]` | every second row | rows 0 and 2 | (2, 4) |
| `m[:, [1]]` | column 1 kept as a column | [[1], [5], [9]] | (3, 1) |



**Watch the shapes.** A single number drops that dimension, while a list or slice keeps it. Many "shape mismatch" errors come from (3,) versus (3, 1).



**The stop value is excluded**, just like Python lists: `m[0:2]` gives rows 0 and 1.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def slicing_shapes():
    m = np.arange(12).reshape(3, 4)
    cell = m[1, 2]
    row = m[0]
    column = m[:, 1]
    block = m[1:, 2:]
    alternate_rows = m[::2]
    column_kept_2d = m[:, [1]]
    shapes = (row.shape, column.shape, block.shape, column_kept_2d.shape)


slicing_shapes()

**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q6](#q6), [Q9](#q9), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 7: Views and copies: a bug worth knowing

*Level: Core*

Slicing doesn't copy the data. It gives a **view**: a new window onto the same numbers.



```python
a = np.array([1, 2, 3, 4])
b = a[1:3]        # a view
b[0] = 99
# a is now [1, 99, 3, 4]
```



That's fast, but it can change your original data without you noticing.



| Operation | Gives |
| --- | --- |
| `a[1:3]`, `a.reshape(...)`, `a.T` | view (shares memory) |
| `a[a > 1]`, `a[[0, 2]]` | copy |
| `a.copy()` | copy, on purpose |



**Rule:** if you plan to change a slice and need the original intact, write `.copy()`.



**Assigning through a mask** writes to the original, and that's usually what you want: `a[a < 0] = 0`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [8]:
def view_copy_demo():
    a = np.array([1, 2, 3, 4])
    view = a[1:3]
    view[0] = 99
    a_after_view = a.copy()
    safe = a[1:3].copy()
    safe[0] = -1
    a_after_copy = a.copy()
    picked = a[a > 2]
    shares = (np.shares_memory(a, view), np.shares_memory(a, safe), np.shares_memory(a, picked))


view_copy_demo()

**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 8: Boolean masks and fancy indexing

*Level: Advanced*

Comparing an array gives an array of True/False — a **mask**. Indexing with a mask keeps the elements where it's True:



```python
scores = np.array([45, 82, 67, 91])
mask = scores >= 60            # [False  True  True  True]
scores[mask]                   # [82 67 91]
scores[(scores > 50) & (scores < 90)]   # combine with & | ~  (and brackets!)
np.where(scores >= 60, "pass", "fail")
(scores >= 60).sum()           # count: True counts as 1
```



**Fancy indexing** picks elements by a list of positions: `scores[[0, 3]]` → `[45 91]`. Unlike slices, masks and fancy indexing return **copies**.



**⚠️ Watch out:** Use `&`, `|`, `~` with arrays, not `and`, `or`, `not` — and put each comparison in brackets.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [9]:
def mask_demo():
    scores = np.array([45, 82, 67, 91])
    passed = scores >= 60
    good = scores[passed]
    middle = scores[(scores > 50) & (scores < 90)]
    labels = np.where(passed, "pass", "fail")
    picked = scores[[0, 3]]
    print(good, middle, labels, picked, passed.sum())


mask_demo()

[82 67 91] [82 67] ['fail' 'pass' 'pass' 'pass'] [45 91] 3


**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q11](#q11), [Q18](#q18), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 9: Reshaping and combining

*Level: Advanced*

| Operation | Example |
| --- | --- |
| Reshape (same data, new shape) | `np.arange(6).reshape(2, 3)`; use −1 for "work it out" |
| Flatten to 1-D | `m.ravel()` (view if possible), `m.flatten()` (copy) |
| Transpose | `m.T` |
| Add an axis | `a[:, np.newaxis]` or `a.reshape(-1, 1)` (column vector) |
| Stack vertically / horizontally | `np.vstack([a, b])`, `np.hstack([a, b])` |
| Join along an axis | `np.concatenate([a, b], axis=0)` |
| Unique values | `np.unique(a, return_counts=True)` |



Reshape never changes the number of elements — `reshape(4, 2)` on 6 elements raises an error.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [10]:
def reshape_demo():
    a = np.arange(6)
    grid = a.reshape(2, 3)
    tall = a.reshape(-1, 1)
    flipped = grid.T
    both = np.vstack([grid, grid * 10])
    values, counts = np.unique([3, 1, 3, 2, 3], return_counts=True)
    print(grid, flipped, both, values, counts, sep="\n")


reshape_demo()

[[0 1 2]
 [3 4 5]]
[[0 3]
 [1 4]
 [2 5]]
[[ 0  1  2]
 [ 3  4  5]
 [ 0 10 20]
 [30 40 50]]
[1 2 3]
[1 1 3]


**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q14](#q14), [Q15](#q15), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 10: Masks, where and -1, step by step

*Level: Advanced*

#### Building a mask



Marks are [35, 72, 90, 48].



1. `marks >= 50` gives [False, True, True, False].
2. `marks[marks >= 50]` gives [72, 90].
3. `(marks >= 50).sum()` gives 2, because True counts as 1.
4. `(marks >= 50).mean()` gives 0.5: half passed.



#### Combining conditions



Use `&` (and), `|` (or), `~` (not), with **brackets around each part**:



```python
marks[(marks >= 50) & (marks < 80)]    # [72]
```



Without brackets, or using the word `and`, Python raises an error.



#### np.where is if/else for a whole array



`np.where(marks >= 50, "pass", "fail")` gives one answer per element.



#### Reshape with -1



-1 means "work this out for me". With 12 values, `reshape(3, -1)` gives 3 rows of 4. `reshape(5, -1)` fails, because 12 doesn't divide by 5.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [11]:
def mask_steps():
    marks = np.array([35, 72, 90, 48])
    passed_mask = marks >= 50
    passed = marks[passed_mask]
    how_many = passed_mask.sum()
    share = passed_mask.mean()
    middle = marks[(marks >= 50) & (marks < 80)]
    labels = np.where(marks >= 50, "pass", "fail")
    rows = np.arange(12).reshape(3, -1)


mask_steps()

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q14](#q14), [Q18](#q18), [Q17](#q17). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Double Every Number

🟢 Warm-up · *Vectorised arithmetic*

Given a list of numbers, return every number doubled.

**Examples**

- `nums = [1, 2, 3]` → `[2, 4, 6]`
- `nums = []` → `[]`
- `nums = [-1.5, 4]` → `[-3.0, 8]`

In [ ]:
def double(nums):
    """
    Given a list of numbers, return every number doubled.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(double, [
    (([1, 2, 3],), [2, 4, 6]),
    (([],), []),
    (([-1.5, 4],), [-3.0, 8]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A Python loop that multiplies each item and appends it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Turn the list into an array; `arr * 2` multiplies every element at once in fast compiled code.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A Python loop that multiplies each item and appends it.

```python
def double(nums):
    out = []
    for x in nums:
        out.append(x * 2)
    return out
```

**Cost:** O(n) — interpreted loop time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Turn the list into an array; `arr * 2` multiplies every element at once in fast compiled code.

```python
def double(nums):
    arr = np.array(nums)
    return arr * 2
```

**Cost:** O(n) — vectorised time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A Python loop that multiplies each item and appends it. That costs O(n) — interpreted loop time and O(n) space."

"I can do better: Turn the list into an array; `arr * 2` multiplies every element at once in fast compiled code. That's O(n) — vectorised time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/double-every-number)

<a id="q2"></a>
### Q2. Shape and Size

🟢 Warm-up · *shape · size · ndim*

Given a rectangular nested list (a table), return `(rows, columns, total_items)`.

**Examples**

- `table = [[1, 2, 3], [4, 5, 6]]` → `(2, 3, 6)`
- `table = [[7]]` → `(1, 1, 1)`

In [ ]:
def shape_size(table):
    """
    Given a rectangular nested list (a table), return (rows, columns,
    total_items).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shape_size, [
    (([[1, 2, 3], [4, 5, 6]],), (2, 3, 6)),
    (([[7]],), (1, 1, 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count rows with len, columns with len of the first row, multiply.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr.shape` is (rows, cols) and `arr.size` is the total.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Count rows with len, columns with len of the first row, multiply.

```python
def shape_size(table):
    rows = len(table)
    cols = len(table[0])
    total = rows * cols
    return (rows, cols, total)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr.shape` is (rows, cols) and `arr.size` is the total.

```python
def shape_size(table):
    arr = np.array(table)
    rows, cols = arr.shape
    return (rows, cols, arr.size)
```

**Cost:** O(n) to build time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count rows with len, columns with len of the first row, multiply. That costs O(1) time and O(1) space."

"I can do better: `arr.shape` is (rows, cols) and `arr.size` is the total. That's O(n) to build time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/shape-and-size)

<a id="q3"></a>
### Q3. Evenly Spaced Numbers

🟢 Warm-up · *linspace*

Return `n` numbers evenly spaced from `start` to `stop` (both included), rounded to 2 decimals.

**Examples**

- `start = 0, stop = 1, n = 5` → `[0.0, 0.25, 0.5, 0.75, 1.0]`
- `start = 10, stop = 20, n = 3` → `[10.0, 15.0, 20.0]`

In [ ]:
def spaced(start, stop, n):
    """
    Return n numbers evenly spaced from start to stop (both included), rounded
    to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(spaced, [
    ((0, 1, 5), [0.0, 0.25, 0.5, 0.75, 1.0]),
    ((10, 20, 3), [10.0, 15.0, 20.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Work out the step (stop − start) / (n − 1) and add it in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`np.linspace(start, stop, n)` does exactly this.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Work out the step (stop − start) / (n − 1) and add it in a loop.

```python
def spaced(start, stop, n):
    step = (stop - start) / (n - 1)
    out = []
    for i in range(n):
        out.append(round(start + i * step, 2))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `np.linspace(start, stop, n)` does exactly this.

```python
def spaced(start, stop, n):
    return np.linspace(start, stop, n).round(2)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Work out the step (stop − start) / (n − 1) and add it in a loop. That costs O(n) time and O(n) space."

"I can do better: `np.linspace(start, stop, n)` does exactly this. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/evenly-spaced-numbers)

<a id="q4"></a>
### Q4. Identity Matrix

🟢 Warm-up · *np.eye*

Return the n × n identity matrix (1 on the diagonal, 0 elsewhere) as integers.

**Examples**

- `n = 3` → `[[1, 0, 0], [0, 1, 0], [0, 0, 1]]`
- `n = 1` → `[[1]]`

In [ ]:
def identity(n):
    """
    Return the n × n identity matrix (1 on the diagonal, 0 elsewhere) as
    integers.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(identity, [
    ((3,), [[1, 0, 0], [0, 1, 0], [0, 0, 1]]),
    ((1,), [[1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops: 1 when row == column.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`np.eye(n, dtype=int)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Nested loops: 1 when row == column.

```python
def identity(n):
    rows = []
    for r in range(n):
        row = []
        for c in range(n):
            row.append(1 if r == c else 0)
        rows.append(row)
    return rows
```

**Cost:** O(n²) time, O(n²) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `np.eye(n, dtype=int)`.

```python
def identity(n):
    return np.eye(n, dtype=int)
```

**Cost:** O(n²) time, O(n²) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops: 1 when row == column. That costs O(n²) time and O(n²) space."

"I can do better: `np.eye(n, dtype=int)`. That's O(n²) time and O(n²) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/identity-matrix)

<a id="q5"></a>
### Q5. Read One Cell

🟢 Warm-up · *2-D indexing*

Return the value at row `r`, column `c` of a table.

**Examples**

- `table = [[1, 2], [3, 4]], r = 1, c = 0` → `3`
- `table = [[5, 6, 7]], r = 0, c = 2` → `7`

In [ ]:
def cell(table, r, c):
    """
    Return the value at row r, column c of a table.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cell, [
    (([[1, 2], [3, 4]], 1, 0), 3),
    (([[5, 6, 7]], 0, 2), 7),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Index twice: `table[r][c]`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy takes both indices at once: `arr[r, c]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Index twice: `table[r][c]`.

```python
def cell(table, r, c):
    row = table[r]
    return row[c]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** NumPy takes both indices at once: `arr[r, c]`.

```python
def cell(table, r, c):
    arr = np.array(table)
    return arr[r, c]
```

**Cost:** O(1) time, O(n) to build space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Index twice: `table[r][c]`. That costs O(1) time and O(1) space."

"I can do better: NumPy takes both indices at once: `arr[r, c]`. That's O(1) time and O(n) to build space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/read-one-cell)

<a id="q6"></a>
### Q6. Last Column

🟢 Warm-up · *Column slicing*

Return the last column of a table as a list.

**Examples**

- `table = [[1, 2, 3], [4, 5, 6]]` → `[3, 6]`
- `table = [[9]]` → `[9]`

In [ ]:
def last_column(table):
    """
    Return the last column of a table as a list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(last_column, [
    (([[1, 2, 3], [4, 5, 6]],), [3, 6]),
    (([[9]],), [9]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over rows and take each row's last item.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr[:, -1]` — all rows, last column.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop over rows and take each row's last item.

```python
def last_column(table):
    col = []
    for row in table:
        col.append(row[-1])
    return col
```

**Cost:** O(rows) time, O(rows) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr[:, -1]` — all rows, last column.

```python
def last_column(table):
    arr = np.array(table)
    return arr[:, -1]
```

**Cost:** O(rows) time, O(rows) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over rows and take each row's last item. That costs O(rows) time and O(rows) space."

"I can do better: `arr[:, -1]` — all rows, last column. That's O(rows) time and O(rows) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/last-column)

<a id="q7"></a>
### Q7. Reverse an Array

🟢 Warm-up · *Slicing with step −1*

Return the numbers in reverse order.

**Examples**

- `nums = [1, 2, 3]` → `[3, 2, 1]`
- `nums = []` → `[]`

In [ ]:
def reverse(nums):
    """
    Return the numbers in reverse order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse, [
    (([1, 2, 3],), [3, 2, 1]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop from the last index down to 0.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr[::-1]` — a reversed view, no copying loop.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop from the last index down to 0.

```python
def reverse(nums):
    out = []
    for i in range(len(nums) - 1, -1, -1):
        out.append(nums[i])
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr[::-1]` — a reversed view, no copying loop.

```python
def reverse(nums):
    return np.array(nums)[::-1]
```

**Cost:** O(1) view time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop from the last index down to 0. That costs O(n) time and O(n) space."

"I can do better: `arr[::-1]` — a reversed view, no copying loop. That's O(1) view time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/reverse-an-array)

<a id="q8"></a>
### Q8. Every Second Element

🟢 Warm-up · *Step slicing*

Return the elements at positions 0, 2, 4, …

**Examples**

- `nums = [10, 11, 12, 13, 14]` → `[10, 12, 14]`
- `nums = [5]` → `[5]`

In [ ]:
def every_second(nums):
    """
    Return the elements at positions 0, 2, 4, …
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(every_second, [
    (([10, 11, 12, 13, 14],), [10, 12, 14]),
    (([5],), [5]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over indices with a step of 2.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr[::2]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop over indices with a step of 2.

```python
def every_second(nums):
    out = []
    for i in range(0, len(nums), 2):
        out.append(nums[i])
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr[::2]`.

```python
def every_second(nums):
    return np.array(nums)[::2]
```

**Cost:** O(1) view time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over indices with a step of 2. That costs O(n) time and O(n) space."

"I can do better: `arr[::2]`. That's O(1) view time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/every-second-element)

<a id="q9"></a>
### Q9. Top-Left Block

🟢 Warm-up · *2-D slicing*

Return the top-left k × k block of a table.

**Examples**

- `table = [[1, 2, 3], [4, 5, 6], [7, 8, 9]], k = 2` → `[[1, 2], [4, 5]]`
- `table = [[1, 2], [3, 4]], k = 1` → `[[1]]`

In [ ]:
def top_left(table, k):
    """
    Return the top-left k × k block of a table.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_left, [
    (([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 2), [[1, 2], [4, 5]]),
    (([[1, 2], [3, 4]], 1), [[1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops over the first k rows and columns.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr[:k, :k]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Nested loops over the first k rows and columns.

```python
def top_left(table, k):
    block = []
    for r in range(k):
        row = []
        for c in range(k):
            row.append(table[r][c])
        block.append(row)
    return block
```

**Cost:** O(k²) time, O(k²) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr[:k, :k]`.

```python
def top_left(table, k):
    arr = np.array(table)
    return arr[:k, :k]
```

**Cost:** O(1) view time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops over the first k rows and columns. That costs O(k²) time and O(k²) space."

"I can do better: `arr[:k, :k]`. That's O(1) view time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/top-left-block)

<a id="q10"></a>
### Q10. Count Values Above a Threshold

🟢 Warm-up · *Boolean arrays*

Return how many numbers are greater than `t`.

**Examples**

- `nums = [3, 8, 1, 9, 5], t = 4` → `3`
- `nums = [1, 2], t = 10` → `0`

In [ ]:
def count_above(nums, t):
    """
    Return how many numbers are greater than t.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_above, [
    (([3, 8, 1, 9, 5], 4), 3),
    (([1, 2], 10), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and count.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr > t` gives an array of True/False; `.sum()` counts the Trues.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop and count.

```python
def count_above(nums, t):
    count = 0
    for x in nums:
        if x > t:
            count += 1
    return count
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr > t` gives an array of True/False; `.sum()` counts the Trues.

```python
def count_above(nums, t):
    arr = np.array(nums)
    mask = arr > t
    return mask.sum()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and count. That costs O(n) time and O(1) space."

"I can do better: `arr > t` gives an array of True/False; `.sum()` counts the Trues. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/count-values-above-a-threshold)

<a id="q11"></a>
### Q11. Replace Negatives With Zero

🟢 Warm-up · *np.where*

Return the numbers with every negative value replaced by 0.

**Examples**

- `nums = [3, -1, 4, -5]` → `[3, 0, 4, 0]`
- `nums = []` → `[]`

In [ ]:
def no_negatives(nums):
    """
    Return the numbers with every negative value replaced by 0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(no_negatives, [
    (([3, -1, 4, -5],), [3, 0, 4, 0]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with an if.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`np.where(arr < 0, 0, arr)` picks 0 where the condition holds, otherwise the value. (`np.maximum(arr, 0)` also works.)

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop with an if.

```python
def no_negatives(nums):
    out = []
    for x in nums:
        if x < 0:
            out.append(0)
        else:
            out.append(x)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `np.where(arr < 0, 0, arr)` picks 0 where the condition holds, otherwise the value. (`np.maximum(arr, 0)` also works.)

```python
def no_negatives(nums):
    arr = np.array(nums)
    return np.where(arr < 0, 0, arr)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with an if. That costs O(n) time and O(n) space."

"I can do better: `np.where(arr < 0, 0, arr)` picks 0 where the condition holds, otherwise the value. (`np.maximum(arr, 0)` also works.) That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/replace-negatives-with-zero)

<a id="q12"></a>
### Q12. Floats to Whole Numbers

🟢 Warm-up · *astype*

Convert decimal numbers to whole numbers by dropping the decimals (2.9 → 2, −2.9 → −2).

**Examples**

- `nums = [2.9, -2.9, 5.0]` → `[2, -2, 5]`
- `nums = [0.5]` → `[0]`

In [ ]:
def to_ints(nums):
    """
    Convert decimal numbers to whole numbers by dropping the decimals (2.9 → 2,
    −2.9 → −2).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_ints, [
    (([2.9, -2.9, 5.0],), [2, -2, 5]),
    (([0.5],), [0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Call `int()` on each number.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr.astype(int)` converts the whole array's dtype (truncating toward zero).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Call `int()` on each number.

```python
def to_ints(nums):
    out = []
    for x in nums:
        out.append(int(x))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr.astype(int)` converts the whole array's dtype (truncating toward zero).

```python
def to_ints(nums):
    return np.array(nums).astype(int)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Call `int()` on each number. That costs O(n) time and O(n) space."

"I can do better: `arr.astype(int)` converts the whole array's dtype (truncating toward zero). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/floats-to-whole-numbers)

<a id="q13"></a>
### Q13. Flatten a Table

🟢 Warm-up · *ravel / flatten*

Turn a 2-D table into one flat list, row by row.

**Examples**

- `table = [[1, 2], [3, 4], [5, 6]]` → `[1, 2, 3, 4, 5, 6]`

In [ ]:
def flatten(table):
    """
    Turn a 2-D table into one flat list, row by row.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(flatten, [
    (([[1, 2], [3, 4], [5, 6]],), [1, 2, 3, 4, 5, 6]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr.ravel()` (a view when possible) or `arr.flatten()` (always a copy).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Two nested loops.

```python
def flatten(table):
    flat = []
    for row in table:
        for x in row:
            flat.append(x)
    return flat
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr.ravel()` (a view when possible) or `arr.flatten()` (always a copy).

```python
def flatten(table):
    return np.array(table).ravel()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two nested loops. That costs O(n) time and O(n) space."

"I can do better: `arr.ravel()` (a view when possible) or `arr.flatten()` (always a copy). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/flatten-a-table)

<a id="q14"></a>
### Q14. Reshape Into Rows

🟢 Warm-up · *reshape*

Arrange a flat list of numbers into a table with `rows` rows (it always divides evenly).

**Examples**

- `nums = [1, 2, 3, 4, 5, 6], rows = 2` → `[[1, 2, 3], [4, 5, 6]]`
- `nums = [1, 2, 3, 4], rows = 4` → `[[1], [2], [3], [4]]`

In [ ]:
def reshape_rows(nums, rows):
    """
    Arrange a flat list of numbers into a table with rows rows (it always
    divides evenly).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reshape_rows, [
    (([1, 2, 3, 4, 5, 6], 2), [[1, 2, 3], [4, 5, 6]]),
    (([1, 2, 3, 4], 4), [[1], [2], [3], [4]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the row length and slice the list row by row.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr.reshape(rows, -1)`; −1 tells NumPy to work out the column count.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Compute the row length and slice the list row by row.

```python
def reshape_rows(nums, rows):
    cols = len(nums) // rows
    table = []
    for r in range(rows):
        table.append(nums[r * cols:(r + 1) * cols])
    return table
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr.reshape(rows, -1)`; −1 tells NumPy to work out the column count.

```python
def reshape_rows(nums, rows):
    return np.array(nums).reshape(rows, -1)
```

**Cost:** O(1) view time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the row length and slice the list row by row. That costs O(n) time and O(n) space."

"I can do better: `arr.reshape(rows, -1)`; −1 tells NumPy to work out the column count. That's O(1) view time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/reshape-into-rows)

<a id="q15"></a>
### Q15. Stack Two Rows

🟢 Warm-up · *vstack / hstack*

Put two equal-length lists on top of each other as a 2-row table.

**Examples**

- `a = [1, 2, 3], b = [4, 5, 6]` → `[[1, 2, 3], [4, 5, 6]]`

In [ ]:
def stack_rows(a, b):
    """
    Put two equal-length lists on top of each other as a 2-row table.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(stack_rows, [
    (([1, 2, 3], [4, 5, 6]), [[1, 2, 3], [4, 5, 6]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a list containing copies of both lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`np.vstack([a, b])` stacks vertically (`hstack` joins side by side).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Build a list containing copies of both lists.

```python
def stack_rows(a, b):
    return [list(a), list(b)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `np.vstack([a, b])` stacks vertically (`hstack` joins side by side).

```python
def stack_rows(a, b):
    return np.vstack([a, b])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a list containing copies of both lists. That costs O(n) time and O(n) space."

"I can do better: `np.vstack([a, b])` stacks vertically (`hstack` joins side by side). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/stack-two-rows)

<a id="q16"></a>
### Q16. Main Diagonal

🟢 Warm-up · *np.diag*

Return the values on the main diagonal of a square table.

**Examples**

- `table = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]` → `[1, 5, 9]`
- `table = [[4]]` → `[4]`

In [ ]:
def diagonal(table):
    """
    Return the values on the main diagonal of a square table.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(diagonal, [
    (([[1, 2, 3], [4, 5, 6], [7, 8, 9]],), [1, 5, 9]),
    (([[4]],), [4]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop i and take `table[i][i]`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`np.diag(arr)` (also `arr.diagonal()`).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop i and take `table[i][i]`.

```python
def diagonal(table):
    out = []
    for i in range(len(table)):
        out.append(table[i][i])
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `np.diag(arr)` (also `arr.diagonal()`).

```python
def diagonal(table):
    return np.diag(np.array(table))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop i and take `table[i][i]`. That costs O(n) time and O(n) space."

"I can do better: `np.diag(arr)` (also `arr.diagonal()`). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/main-diagonal)

<a id="q17"></a>
### Q17. Position of the Largest Value

🟢 Warm-up · *argmax*

Return the index of the largest number (first one on ties).

**Examples**

- `nums = [4, 9, 2, 9]` → `1`
- `nums = [-3, -1]` → `1`

In [ ]:
def argmax(nums):
    """
    Return the index of the largest number (first one on ties).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(argmax, [
    (([4, 9, 2, 9],), 1),
    (([-3, -1],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the best index in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`arr.argmax()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Track the best index in a loop.

```python
def argmax(nums):
    best = 0
    for i in range(1, len(nums)):
        if nums[i] > nums[best]:
            best = i
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `arr.argmax()`.

```python
def argmax(nums):
    return np.array(nums).argmax()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the best index in a loop. That costs O(n) time and O(1) space."

"I can do better: `arr.argmax()`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/position-of-the-largest-value)

<a id="q18"></a>
### Q18. Pick Items by Position

🟢 Warm-up · *Fancy indexing*

Return the items at the given list of positions, in that order.

**Examples**

- `nums = [10, 20, 30, 40], positions = [3, 0, 3]` → `[40, 10, 40]`
- `nums = [5, 6], positions = []` → `[]`

In [ ]:
def pick(nums, positions):
    """
    Return the items at the given list of positions, in that order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pick, [
    (([10, 20, 30, 40], [3, 0, 3]), [40, 10, 40]),
    (([5, 6], []), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over the positions.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Index with a list of positions: `arr[positions]`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop over the positions.

```python
def pick(nums, positions):
    out = []
    for p in positions:
        out.append(nums[p])
    return out
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Index with a list of positions: `arr[positions]`.

```python
def pick(nums, positions):
    return np.array(nums)[np.array(positions, dtype=int)]
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over the positions. That costs O(k) time and O(k) space."

"I can do better: Index with a list of positions: `arr[positions]`. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/pick-items-by-position)

<a id="q19"></a>
### Q19. Unique Values and Their Counts

🟡 Interview standard · *np.unique*

Return `(sorted unique values, how many times each appears)`.

**Examples**

- `nums = [3, 1, 3, 2, 1, 3]` → `([1, 2, 3], [2, 1, 3])`
- `nums = [5]` → `([5], [1])`

In [ ]:
def unique_counts(nums):
    """
    Return (sorted unique values, how many times each appears).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unique_counts, [
    (([3, 1, 3, 2, 1, 3],), ([1, 2, 3], [2, 1, 3])),
    (([5],), ([5], [1])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count with a dict, then sort the keys.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`np.unique(arr, return_counts=True)` returns both arrays.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Count with a dict, then sort the keys.

```python
def unique_counts(nums):
    counts = {}
    for x in nums:
        counts[x] = counts.get(x, 0) + 1
    values = sorted(counts)
    return (values, [counts[v] for v in values])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `np.unique(arr, return_counts=True)` returns both arrays.

```python
def unique_counts(nums):
    values, counts = np.unique(nums, return_counts=True)
    return (values, counts)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count with a dict, then sort the keys. That costs O(n log n) time and O(n) space."

"I can do better: `np.unique(arr, return_counts=True)` returns both arrays. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/unique-values-and-their-counts)

<a id="q20"></a>
### Q20. Checkerboard

🟡 Interview standard · *Index arithmetic*

Return an n × n checkerboard of 0s and 1s with 0 in the top-left corner.

**Examples**

- `n = 3` → `[[0, 1, 0], [1, 0, 1], [0, 1, 0]]`
- `n = 1` → `[[0]]`

In [ ]:
def checkerboard(n):
    """
    Return an n × n checkerboard of 0s and 1s with 0 in the top-left corner.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(checkerboard, [
    ((3,), [[0, 1, 0], [1, 0, 1], [0, 1, 0]]),
    ((1,), [[0]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops with (row + col) % 2.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`np.indices((n, n)).sum(axis=0) % 2` builds every row+col sum at once.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Nested loops with (row + col) % 2.

```python
def checkerboard(n):
    board = []
    for r in range(n):
        board.append([(r + c) % 2 for c in range(n)])
    return board
```

**Cost:** O(n²) time, O(n²) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `np.indices((n, n)).sum(axis=0) % 2` builds every row+col sum at once.

```python
def checkerboard(n):
    rows, cols = np.indices((n, n))
    return (rows + cols) % 2
```

**Cost:** O(n²) time, O(n²) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops with (row + col) % 2. That costs O(n²) time and O(n²) space."

"I can do better: `np.indices((n, n)).sum(axis=0) % 2` builds every row+col sum at once. That's O(n²) time and O(n²) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/numpy-basics/checkerboard)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why is NumPy faster than lists?</b></summary>

Arrays store raw values of one type contiguously, and operations run as compiled loops (often SIMD) over that memory instead of interpreting Python bytecode and unboxing objects for each element.

</details>

<details>
<summary><b>2. What is the difference between shape (3,) and (3, 1)?</b></summary>

(3,) is a 1-D array of 3 elements. (3, 1) is a 2-D column with 3 rows and 1 column. They broadcast differently and are not interchangeable.

</details>

<details>
<summary><b>3. View vs copy?</b></summary>

A view shares data with the original array (slices, reshape, transpose), so changes show in both. A copy has its own data (masks, fancy indexing, `.copy()`).

</details>

<details>
<summary><b>4. Why does np.array([1, 2, 'a']) become all strings?</b></summary>

An array has a single dtype; NumPy upcasts to a type that can represent every value, here a Unicode string.

</details>

<details>
<summary><b>5. arange vs linspace?</b></summary>

`arange(start, stop, step)` uses a step and excludes stop (like range). `linspace(start, stop, num)` uses a count and includes stop — better for floats.

</details>

<details>
<summary><b>6. ravel vs flatten?</b></summary>

Both return a 1-D array. `ravel` returns a view when possible; `flatten` always returns a copy.

</details>

<details>
<summary><b>7. What does axis=0 mean?</b></summary>

Operate down the rows (along the first axis), producing one result per column. axis=1 operates across the columns, one result per row.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why NumPy?
- [ ] I can explain: Creating arrays
- [ ] I can explain: How arrays are stored, and why dtype matters
- [ ] I can explain: Shape, dimensions and dtype
- [ ] I can explain: Indexing and slicing
- [ ] I can explain: Reading 2-D slices, worked examples
- [ ] I can explain: Views and copies: a bug worth knowing
- [ ] I can explain: Boolean masks and fancy indexing
- [ ] I can explain: Reshaping and combining
- [ ] I can explain: Masks, where and -1, step by step

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q19](#q19) Unique Values and Their Counts
- [ ] [Q20](#q20) Checkerboard

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.